# Module 00 — Notebook 02: Your First Model Explanation

## Learning Objectives
By completing this notebook, you will:
1. Explain an ImageNet prediction using Integrated Gradients in under 2 minutes
2. Interpret what the attribution heatmap tells you about the model's reasoning
3. Identify the key parameters in the attribution API
4. Recognize the difference between attribution and prediction

## Prerequisites
- Notebook 01 completed (environment verified)
- Basic understanding of neural network forward pass

## Estimated Time: 12 minutes

---

## The Complete Pipeline in 8 Lines

Before any explanation, here is the complete attribution pipeline:

```
1. Load pretrained model
2. Preprocess input image
3. Get model prediction (what class?)
4. Compute attribution (why that class?)
5. Visualize attribution
6. Interpret result
```

Every notebook in this course follows this pattern. This notebook makes it explicit.

## Step 1: Setup

In [ ]:
# Core imports
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import io
import json

# Captum
from captum.attr import IntegratedGradients
from captum.attr import visualization as viz

# Reproducibility
torch.manual_seed(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {DEVICE}")

## Step 2: Load a Pretrained Model

We use ResNet-50 pretrained on ImageNet. This model classifies images into 1000 categories.

**Critical:** Always call `model.eval()` before attribution. This disables dropout and sets batch normalization to use its running statistics rather than batch statistics. Without this, attributions are non-deterministic.

In [ ]:
# Load ResNet-50 pretrained on ImageNet
# weights='IMAGENET1K_V1' specifies the training checkpoint
model = models.resnet50(weights='IMAGENET1K_V1')
model = model.to(DEVICE)

# CRITICAL: eval() before ANY attribution
# - Disables dropout (deterministic outputs)
# - Uses stored running mean/var in BatchNorm (stable gradients)
model.eval()

print("ResNet-50 loaded and set to eval mode.")
print(f"Parameter count: {sum(p.numel() for p in model.parameters()):,}")
print(f"Output classes: 1000 (ImageNet)")

## Step 3: Load and Preprocess the Image

ImageNet models expect:
- 224×224 pixel RGB images
- Pixel values normalized using ImageNet statistics: mean `[0.485, 0.456, 0.406]`, std `[0.229, 0.224, 0.225]`
- A batch dimension: shape `(1, 3, 224, 224)`

In [ ]:
# ImageNet preprocessing pipeline
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(tensor):
    """Undo ImageNet normalization for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

# Load a sample image
# Using a dog photo from Wikimedia Commons (public domain)
IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"

print("Loading image...")
try:
    with urllib.request.urlopen(IMAGE_URL) as resp:
        raw_image = Image.open(io.BytesIO(resp.read())).convert('RGB')
    print(f"Image loaded: {raw_image.size}")
except Exception:
    # Fallback: generate a synthetic image if download fails
    print("Download failed. Using synthetic image.")
    arr = np.zeros((320, 320, 3), dtype=np.uint8)
    # Simple dog-like pattern (just for pipeline testing)
    arr[80:240, 80:240, :] = [180, 140, 80]
    arr[60:100, 90:230, :] = [160, 120, 70]
    raw_image = Image.fromarray(arr)

# Preprocess
# unsqueeze(0) adds the batch dimension: (3,224,224) → (1,3,224,224)
input_tensor = preprocess(raw_image).unsqueeze(0).to(DEVICE)

print(f"Preprocessed tensor shape: {input_tensor.shape}")
print(f"Value range: [{input_tensor.min():.2f}, {input_tensor.max():.2f}]")

# Display the image
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
display_img = denormalize(input_tensor.squeeze(0).cpu()).permute(1, 2, 0).numpy()
ax.imshow(display_img)
ax.set_title("Input Image (224×224)")
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 4: Get the Model's Prediction

Before computing attributions, we need to know *what* the model predicted. Attributions explain the model's reasoning for a specific output class — usually the top predicted class.

In [ ]:
# Download ImageNet class labels
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        imagenet_labels = json.loads(resp.read().decode())
except Exception:
    imagenet_labels = [f"class_{i}" for i in range(1000)]
    imagenet_labels[207] = "golden retriever"
    imagenet_labels[208] = "Labrador retriever"
    imagenet_labels[263] = "Pembroke Welsh corgi"

# Forward pass to get the prediction
# torch.no_grad(): we only need outputs here, not gradients
with torch.no_grad():
    logits = model(input_tensor)
    probabilities = torch.softmax(logits, dim=1)

# Top-5 predictions
top5_probs, top5_indices = probabilities[0].topk(5)

print("Top-5 ImageNet Predictions")
print("=" * 45)
for prob, idx in zip(top5_probs, top5_indices):
    print(f"  [{idx.item():4d}] {imagenet_labels[idx.item()]:<35} {prob.item():.2%}")

# The class we will explain
top_class = top5_indices[0].item()
top_prob = top5_probs[0].item()
print(f"\nWill explain prediction: '{imagenet_labels[top_class]}' (class {top_class})")

## Step 5: Compute Integrated Gradients Attribution

Integrated Gradients computes attributions by:
1. Starting from a **baseline** (black image = absence of information)
2. Interpolating 50 steps from baseline to input
3. Computing gradients at each interpolation step
4. Integrating (summing) gradients along this path

The result: each pixel gets a score representing how much it contributed to the predicted class.

The complete derivation is in Module 02. For now, focus on *what it produces*.

In [ ]:
# Integrated Gradients attribution

# Step 1: Enable gradients on input
# Captum's gradient methods need input.requires_grad = True
input_for_attr = input_tensor.requires_grad_(True)

# Step 2: Define the baseline
# Zero baseline = black image = "no information" reference point
# The attribution measures: how much does each pixel contribute
# compared to having no image at all?
baseline = torch.zeros_like(input_tensor).to(DEVICE)

# Step 3: Create the Integrated Gradients method
ig = IntegratedGradients(model)

# Step 4: Compute attributions
# n_steps=50: integrate over 50 steps from baseline to input
# target=top_class: explain the top predicted class
# return_convergence_delta=True: also return the approximation error
print("Computing Integrated Gradients...")
attributions, delta = ig.attribute(
    input_for_attr,
    baselines=baseline,
    target=top_class,
    n_steps=50,
    return_convergence_delta=True
)

print(f"\nAttribution shape: {attributions.shape}")
print(f"Same shape as input: {attributions.shape == input_tensor.shape}")
print(f"\nConvergence delta: {delta.item():.5f}")
print("  (delta near 0 = good approximation of the true integral)")
print(f"  (|delta| < 0.05 is generally acceptable)")

# Verify the completeness property:
# sum(attributions) should ≈ f(input) - f(baseline)
with torch.no_grad():
    f_input = model(input_tensor)[0, top_class].item()
    f_baseline = model(baseline)[0, top_class].item()

attr_sum = attributions.sum().item()
expected_sum = f_input - f_baseline

print(f"\nCompleteness check:")
print(f"  Sum of attributions:  {attr_sum:.4f}")
print(f"  f(input) - f(base):   {expected_sum:.4f}")
print(f"  Match: {abs(attr_sum - expected_sum) < 0.1}")

## Step 6: Visualize the Attribution

The attribution tensor has the same shape as the input: `(1, 3, 224, 224)`. Each of the 150,528 values represents how much that pixel (in that color channel) contributed to the predicted class.

We visualize this four ways:
1. The original image
2. The raw attribution heatmap (positive values only)
3. Attribution overlaid on the image
4. The image masked to show only the high-attribution regions

In [ ]:
# Visualize attributions using Captum's visualization module

# Convert to numpy arrays for visualization
# Captum viz expects (H, W, C) format with values in [0, 1]
image_np = denormalize(input_tensor.squeeze(0).cpu()).permute(1, 2, 0).numpy()

# Attribution: (C, H, W) → (H, W, C)
attr_np = attributions.squeeze(0).cpu().permute(1, 2, 0).detach().numpy()

# Four-panel visualization
fig, axes = viz.visualize_image_attr_multiple(
    attr_np,
    image_np,
    methods=[
        "original_image",    # Panel 1: original image
        "heat_map",          # Panel 2: attribution heatmap
        "blended_heat_map",  # Panel 3: heatmap overlaid on image
        "masked_image"       # Panel 4: image × high-attribution mask
    ],
    signs=[
        "all",               # Show original as-is
        "absolute_value",    # Show absolute attribution magnitude
        "absolute_value",    # Show absolute attribution overlaid
        "positive"           # Mask: show only positive attribution regions
    ],
    titles=[
        "Input Image",
        "Attribution Heatmap",
        "Overlay",
        "High-Attribution Regions"
    ],
    show_colorbar=True,
    fig_size=(18, 4),
    use_pyplot=True
)

plt.suptitle(
    f"Integrated Gradients — Predicted: '{imagenet_labels[top_class]}' ({top_prob:.1%} confidence)",
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.show()

## Step 7: Interpret the Attribution

Reading an attribution heatmap:

- **Warm colors (red/yellow):** pixels that *increased* the model's confidence in the predicted class
- **Cool colors (blue/purple):** pixels that *decreased* the model's confidence
- **No color / white:** pixels that had little effect on the prediction

For a well-functioning model predicting a dog:
- The dog's body, face, and distinctive features should be warm
- The background should be mostly uncolored or cool

If the background is warm and the dog is uncolored, the model is using spurious features.

In [ ]:
# Quantitative analysis of the attribution

attr_abs = np.abs(attr_np).mean(axis=-1)  # Mean absolute attribution across channels

# Attribution statistics
total_attr = attr_abs.sum()

# Top 10% of pixels by attribution magnitude
threshold_90 = np.percentile(attr_abs, 90)
top10_mask = attr_abs >= threshold_90
top10_fraction = top10_mask.mean()
top10_attr_share = attr_abs[top10_mask].sum() / total_attr

print("Attribution Statistics")
print("=" * 45)
print(f"Total attribution magnitude: {total_attr:.4f}")
print(f"\nTop 10% pixels (by attribution):")
print(f"  Fraction of image: {top10_fraction:.1%}")
print(f"  Share of total attribution: {top10_attr_share:.1%}")
print(f"\nThis means the top 10% of pixels account for {top10_attr_share:.0%}")
print(f"of the total attribution — attribution is {'concentrated' if top10_attr_share > 0.5 else 'spread out'}.")

# Visualize the attribution distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of attribution values
axes[0].hist(attr_abs.flatten(), bins=100, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(threshold_90, color='red', linestyle='--', label=f'90th percentile')
axes[0].set_xlabel('Attribution Magnitude')
axes[0].set_ylabel('Pixel Count')
axes[0].set_title('Distribution of Attribution Values')
axes[0].legend()
axes[0].set_yscale('log')

# Top-10% attribution mask
axes[1].imshow(image_np)
axes[1].imshow(
    np.ma.masked_where(~top10_mask, attr_abs),
    alpha=0.7, cmap='Reds',
    vmin=threshold_90, vmax=attr_abs.max()
)
axes[1].set_title(f'Top 10% Most Attributed Pixels')
axes[1].axis('off')

plt.suptitle('Attribution Analysis', fontsize=13)
plt.tight_layout()
plt.show()

## Self-Check Exercise

**Task:** Explain the model's prediction for a *different* class than the top-1 prediction.

If ResNet-50 predicted "Labrador retriever" with 85% confidence, the top-2 prediction might be "golden retriever" with 8%. What does the model "look at" differently when considering these two classes?

Compute attributions for the 2nd-ranked class and compare them with the top-1 attributions.

In [ ]:
# Self-check: explain the 2nd-ranked class

# Get the second-ranked class index
second_class = top5_indices[1].item()
second_prob = top5_probs[1].item()

print(f"Top-1 class: {imagenet_labels[top_class]} ({top_prob:.1%})")
print(f"Top-2 class: {imagenet_labels[second_class]} ({second_prob:.1%})")
print("\nComputing attributions for both classes...")

# Compute attributions for top-1
attr_top1, _ = ig.attribute(
    input_for_attr,
    baselines=baseline,
    target=top_class,    # Top-1 class
    n_steps=50,
    return_convergence_delta=True
)

# Compute attributions for top-2
attr_top2, _ = ig.attribute(
    input_for_attr,
    baselines=baseline,
    target=second_class,  # Top-2 class
    n_steps=50,
    return_convergence_delta=True
)

# Prepare numpy arrays
attr1_np = attr_top1.squeeze(0).cpu().permute(1, 2, 0).detach().numpy()
attr2_np = attr_top2.squeeze(0).cpu().permute(1, 2, 0).detach().numpy()

# Side-by-side comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row_idx, (attr_np_curr, class_idx, class_prob) in enumerate([
    (attr1_np, top_class, top_prob),
    (attr2_np, second_class, second_prob)
]):
    class_label = imagenet_labels[class_idx]

    axes[row_idx, 0].imshow(image_np)
    axes[row_idx, 0].set_title(f"Input Image\n(Explaining: {class_label})")
    axes[row_idx, 0].axis('off')

    # Heatmap
    attr_magnitude = np.abs(attr_np_curr).mean(axis=-1)
    im = axes[row_idx, 1].imshow(attr_magnitude, cmap='hot')
    axes[row_idx, 1].set_title(f"Attribution Heatmap\n{class_label} ({class_prob:.1%})")
    axes[row_idx, 1].axis('off')
    plt.colorbar(im, ax=axes[row_idx, 1], fraction=0.046)

    # Overlay
    axes[row_idx, 2].imshow(image_np)
    axes[row_idx, 2].imshow(
        attr_magnitude, alpha=0.5, cmap='hot',
        vmin=np.percentile(attr_magnitude, 80)
    )
    axes[row_idx, 2].set_title(f"Overlay\n{class_label}")
    axes[row_idx, 2].axis('off')

plt.suptitle(
    "Attribution Comparison: Top-1 vs Top-2 Predicted Class",
    fontsize=14
)
plt.tight_layout()
plt.show()

# Attribution overlap analysis
attr1_mag = np.abs(attr1_np).mean(axis=-1)
attr2_mag = np.abs(attr2_np).mean(axis=-1)

# Correlation between the two attribution maps
corr = np.corrcoef(attr1_mag.flatten(), attr2_mag.flatten())[0, 1]
print(f"\nAttribution map correlation (top-1 vs top-2): {corr:.3f}")
print("  Correlation > 0.8: models look at the same regions for both classes")
print("  Correlation < 0.5: models use different image regions for each class")

## Summary

### What You Just Did

You ran a complete neural network explanation pipeline:

1. Loaded a pretrained ResNet-50 (ImageNet, 1000 classes)
2. Preprocessed an image to the correct format
3. Got the model's top-5 predictions
4. Computed Integrated Gradients attribution for the top class
5. Visualized the attribution as a heatmap
6. Validated the completeness property
7. Compared attributions for two different classes

### What You Observed

- Attribution heatmaps show *where* in the image the model focused
- Different target classes produce different attribution patterns
- The completeness property connects attributions to the output score change

### What's Next

**Module 01** explores four gradient-based methods side-by-side — Saliency, Input×Gradient, Guided Backpropagation, and Integrated Gradients — building intuition for when each is appropriate.

**Module 02** provides the complete mathematical derivation of Integrated Gradients, explains the baseline choice, and covers SmoothGrad for variance reduction.

### Key Takeaways

1. `model.eval()` is mandatory before attribution
2. Input tensors need `requires_grad_(True)` for gradient methods
3. The baseline is the reference "no information" point
4. Attribution shape always matches input shape
5. Convergence delta validates approximation quality